# Global Superstore Sales Analysis
**End-to-end Python data analytics project**

## Objective
Analyze a Global Superstore order-level dataset to understand sales, profitability, product performance, customer segments, regional/market patterns, discounts, and shipping.

## Tools
Python, Pandas, NumPy, Matplotlib, Seaborn, Jupyter Notebook / Google Colab.

## Dataset note
This notebook was prepared for the provided `Global-Superstore.csv` file. The source file uses a legacy text encoding, so it is loaded with `encoding="latin1"`. It also contains mostly-empty auxiliary columns; those are reviewed and removed during cleaning.

## Attribution
If this dataset was supplied by an instructor or downloaded from another repository, cite that exact source here and follow its data license/permission terms. Do not claim ownership of the original dataset. Describe your own analysis improvements in the README.

## 1. Import libraries

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## 2. Load the dataset

In [ ]:
DATA_PATH = "Global-Superstore.csv"

# This file is encoded as a legacy single-byte text file, not UTF-8.
df_raw = pd.read_csv(DATA_PATH, encoding="latin1", low_memory=False)
df = df_raw.copy()

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
display(df.head())

## 3. Initial data inspection

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\\nData types and non-null counts:")
df.info()

print("\\nDuplicate rows:", df.duplicated().sum())

missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)
display(missing)
display(df.describe(include="all").T)

## 4. Data cleaning and preprocessing

- Normalize column names.
- Remove completely empty columns and auxiliary columns that are overwhelmingly empty.
- Parse order and ship dates.
- Convert numeric measures to numeric types.
- Remove exact duplicate rows, if any.
- Keep missing postal codes as missing: postal codes are not available for many international records, so filling them with fabricated values would be misleading.

In [ ]:
def normalize_name(column):
    return (
        str(column).strip().lower()
        .replace(" ", "_").replace("-", "_").replace("/", "_")
    )

df.columns = [normalize_name(c) for c in df.columns]

# Remove columns that contain no useful data.
all_empty = [c for c in df.columns if df[c].isna().all()]
df.drop(columns=all_empty, inplace=True)

# Some exports contain stray unnamed/helper columns with almost all values missing.
mostly_empty = [
    c for c in df.columns
    if c.startswith("unnamed") and df[c].isna().mean() >= 0.90
]
df.drop(columns=mostly_empty, inplace=True)

# Drop the mostly-empty derived/helper column if present; it is not needed for core analysis.
if "sales_10%" in df.columns and df["sales_10%"].isna().mean() >= 0.90:
    df.drop(columns=["sales_10%"], inplace=True)

for col in ["order_date", "ship_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

for col in ["row_id", "postal_code", "sales", "quantity", "discount", "profit", "shipping_cost"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

rows_before = len(df)
df = df.drop_duplicates().copy()
duplicates_removed = rows_before - len(df)

print("Removed all-empty columns:", all_empty)
print("Removed mostly-empty helper columns:", mostly_empty)
print("Exact duplicate rows removed:", duplicates_removed)
print("Cleaned shape:", df.shape)
display(df.head())

## 5. Data quality after cleaning

In [ ]:
post_cleaning_missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)
display(post_cleaning_missing)

if "order_date" in df.columns:
    print("Invalid/missing order dates:", df["order_date"].isna().sum())
if "ship_date" in df.columns:
    print("Invalid/missing ship dates:", df["ship_date"].isna().sum())

# Negative profit can be a real business outcome, so do not delete it as an error.
if "profit" in df.columns:
    print("Rows with negative profit:", int((df["profit"] < 0).sum()))

## 6. Feature engineering

In [ ]:
if "order_date" in df.columns:
    df["order_year"] = df["order_date"].dt.year
    df["order_month"] = df["order_date"].dt.month
    df["order_year_month"] = df["order_date"].dt.to_period("M").astype(str)

if "order_date" in df.columns and "ship_date" in df.columns:
    df["shipping_days"] = (df["ship_date"] - df["order_date"]).dt.days

if "sales" in df.columns and "profit" in df.columns:
    df["profit_margin_pct"] = np.where(
        df["sales"] != 0, df["profit"] / df["sales"] * 100, np.nan
    )

display(df.head())

## 7. Key performance indicators (KPIs)

In [ ]:
kpis = {}
if "sales" in df.columns:
    kpis["Total Sales"] = df["sales"].sum()
if "profit" in df.columns:
    kpis["Total Profit"] = df["profit"].sum()
if "quantity" in df.columns:
    kpis["Units Sold"] = df["quantity"].sum()
if "order_id" in df.columns:
    kpis["Unique Orders"] = df["order_id"].nunique()
if "customer_id" in df.columns:
    kpis["Unique Customers"] = df["customer_id"].nunique()
if "product_id" in df.columns:
    kpis["Unique Products"] = df["product_id"].nunique()
if "sales" in df.columns and "profit" in df.columns and df["sales"].sum() != 0:
    kpis["Overall Profit Margin (%)"] = df["profit"].sum() / df["sales"].sum() * 100

display(pd.DataFrame({"Metric": list(kpis.keys()), "Value": list(kpis.values())}))

## 8. Sales and profit by category

In [ ]:
if "category" in df.columns and "sales" in df.columns:
    sales_by_category = df.groupby("category")["sales"].sum().sort_values(ascending=False)
    display(sales_by_category.to_frame("total_sales"))
    plt.figure(figsize=(9, 5))
    sns.barplot(x=sales_by_category.values, y=sales_by_category.index)
    plt.title("Total Sales by Category")
    plt.xlabel("Sales")
    plt.ylabel("Category")
    plt.tight_layout()
    plt.show()

if "category" in df.columns and "profit" in df.columns:
    profit_by_category = df.groupby("category")["profit"].sum().sort_values(ascending=False)
    display(profit_by_category.to_frame("total_profit"))
    plt.figure(figsize=(9, 5))
    sns.barplot(x=profit_by_category.values, y=profit_by_category.index)
    plt.title("Total Profit by Category")
    plt.xlabel("Profit")
    plt.ylabel("Category")
    plt.tight_layout()
    plt.show()

## 9. Monthly sales trend

In [ ]:
if {"order_date", "sales"}.issubset(df.columns):
    monthly_sales = (
        df.dropna(subset=["order_date"])
          .set_index("order_date")["sales"]
          .resample("MS").sum()
    )
    display(monthly_sales.to_frame("monthly_sales").head(12))
    plt.figure(figsize=(12, 5))
    monthly_sales.plot()
    plt.title("Monthly Sales Trend")
    plt.xlabel("Month")
    plt.ylabel("Sales")
    plt.tight_layout()
    plt.show()

## 10. Market and regional performance

In [ ]:
for dimension in ["market", "region"]:
    if dimension in df.columns and "sales" in df.columns:
        summary = df.groupby(dimension)["sales"].sum().sort_values(ascending=False)
        display(summary.to_frame("total_sales"))
        plt.figure(figsize=(10, 6))
        sns.barplot(x=summary.values, y=summary.index)
        plt.title(f"Total Sales by {dimension.title()}")
        plt.xlabel("Sales")
        plt.ylabel(dimension.title())
        plt.tight_layout()
        plt.show()

    if dimension in df.columns and "profit" in df.columns:
        display(df.groupby(dimension)["profit"].sum().sort_values().to_frame("total_profit"))

## 11. Top products and sub-categories

In [ ]:
if "product_name" in df.columns and "sales" in df.columns:
    top_products = df.groupby("product_name")["sales"].sum().sort_values(ascending=False).head(10)
    display(top_products.to_frame("total_sales"))
    plt.figure(figsize=(11, 6))
    sns.barplot(x=top_products.values, y=top_products.index)
    plt.title("Top 10 Products by Sales")
    plt.xlabel("Sales")
    plt.ylabel("Product")
    plt.tight_layout()
    plt.show()

if "sub_category" in df.columns and "profit" in df.columns:
    subcategory_profit = df.groupby("sub_category")["profit"].sum().sort_values()
    display(subcategory_profit.to_frame("total_profit"))
    plt.figure(figsize=(10, 7))
    sns.barplot(x=subcategory_profit.values, y=subcategory_profit.index)
    plt.title("Profit by Sub-Category")
    plt.xlabel("Profit")
    plt.ylabel("Sub-Category")
    plt.tight_layout()
    plt.show()

## 12. Customer segment analysis

In [ ]:
if "segment" in df.columns and "sales" in df.columns:
    segment_sales = df.groupby("segment")["sales"].sum().sort_values(ascending=False)
    display(segment_sales.to_frame("total_sales"))
    plt.figure(figsize=(8, 5))
    sns.barplot(x=segment_sales.index, y=segment_sales.values)
    plt.title("Sales by Customer Segment")
    plt.xlabel("Segment")
    plt.ylabel("Sales")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

if "segment" in df.columns and "profit" in df.columns:
    display(df.groupby("segment")["profit"].sum().sort_values(ascending=False).to_frame("total_profit"))

## 13. Discount versus profit

In [ ]:
if {"discount", "profit"}.issubset(df.columns):
    plot_data = df[["discount", "profit"]].dropna()
    if len(plot_data) > 10000:
        plot_data = plot_data.sample(10000, random_state=42)
    plt.figure(figsize=(9, 5))
    sns.scatterplot(data=plot_data, x="discount", y="profit", alpha=0.35)
    plt.title("Discount vs Profit")
    plt.xlabel("Discount")
    plt.ylabel("Profit")
    plt.tight_layout()
    plt.show()

    discount_summary = df.groupby("discount").agg(
        row_count=("profit", "size"),
        total_profit=("profit", "sum"),
        average_profit=("profit", "mean")
    ).sort_index()
    display(discount_summary)

## 14. Shipping analysis

In [ ]:
if "shipping_days" in df.columns:
    print("Rows with negative shipping duration:", int((df["shipping_days"] < 0).sum()))
    display(df["shipping_days"].describe().to_frame("shipping_days_summary"))
    if "ship_mode" in df.columns:
        display(
            df.groupby("ship_mode")["shipping_days"]
              .median().sort_values()
              .to_frame("median_shipping_days")
        )

## 15. Generate data-driven draft insights

In [ ]:
insights = []

if "category" in df.columns and "sales" in df.columns:
    s = df.groupby("category")["sales"].sum().sort_values(ascending=False)
    if not s.empty:
        insights.append(f"Highest-sales category: {s.index[0]} (sales = {s.iloc[0]:,.2f}).")

if "category" in df.columns and "profit" in df.columns:
    p = df.groupby("category")["profit"].sum().sort_values(ascending=False)
    if not p.empty:
        insights.append(f"Highest-profit category: {p.index[0]} (profit = {p.iloc[0]:,.2f}).")

if "market" in df.columns and "sales" in df.columns:
    m = df.groupby("market")["sales"].sum().sort_values(ascending=False)
    if not m.empty:
        insights.append(f"Highest-sales market: {m.index[0]} (sales = {m.iloc[0]:,.2f}).")

if "region" in df.columns and "sales" in df.columns:
    r = df.groupby("region")["sales"].sum().sort_values(ascending=False)
    if not r.empty:
        insights.append(f"Highest-sales region: {r.index[0]} (sales = {r.iloc[0]:,.2f}).")

if "profit" in df.columns:
    insights.append(f"Records with negative profit: {(df['profit'] < 0).sum():,} of {len(df):,} rows.")

for i, insight in enumerate(insights, start=1):
    print(f"{i}. {insight}")

## 16. Export cleaned data

Only include the cleaned dataset in a public repository if the dataset's terms allow redistribution. If not, remove this export from the public repo and add instructions for obtaining the source data.

In [ ]:
cleaned_path = "Global-Superstore-Cleaned.csv"
df.to_csv(cleaned_path, index=False)
print("Saved:", cleaned_path)

## 17. Conclusion and recommendations

Use the actual output above to write your final conclusions. Answer:
1. Which category earns the most sales and profit?
2. Which market and region perform best and worst?
3. What patterns appear in monthly sales?
4. Are higher discounts associated with lower profits in this dataset?
5. Which sub-categories show losses and deserve further investigation?
6. What limitations remain in the data?

Do not claim correlation proves causation. Explain that negative profit may represent real loss-making transactions and should not automatically be removed.